In [1]:
import json, re, pickle, sys
import numpy as np
from collections import defaultdict
from sentence_transformers import SentenceTransformer

sys.path.append("../src")
from llm import generate

docs = pickle.load(open("../data/index/docs.pkl", "rb"))
embeddings = np.load("../data/index/embeddings.npy")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(f"{len(docs)} documents, {embeddings.shape[1]}-dim embeddings")

W1102 20:00:17.056000 85393 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

557 documents, 384-dim embeddings


In [4]:
# --- Lookup tables ---
SEASONS = sorted({d["season"] for d in docs})
PLAYER_SEASONS = sorted({d["season"] for d in docs if d["type"] == "player_season"})

player_idx = defaultdict(set)
for i, d in enumerate(docs):
    if d["type"] == "player_season":
        full = d["player"].lower()
        player_idx[full].add(i)
        player_idx[full.split()[-1]].add(i)

team_idx = defaultdict(set)
for i, d in enumerate(docs):
    if d["type"] == "team_season":
        team_idx[d["team"].lower()].add(i)
        team_idx[d["team"].lower().split()[-1]].add(i)
        team_idx[d["team_abbr"].lower()].add(i)


# --- Query parser ---
def parse_query(q):
    ql = q.lower()

    season = None
    m = re.search(r"(20\d\d)[-–](\d\d)", q)
    if m and f"{m.group(1)}-{m.group(2)}" in SEASONS:
        season = f"{m.group(1)}-{m.group(2)}"
    else:
        m = re.search(r"\b(20\d\d)\b", q)
        if m:
            y = m.group(1)
            for s in SEASONS:
                if s.startswith(y) or s.endswith(y[-2:]):
                    season = s
                    break

    players = {n for n in player_idx if len(n) > 3 and n in ql}
    teams   = {n for n in team_idx   if len(n) > 3 and n in ql}
    return {"season": season, "players": players, "teams": teams}


# --- Hybrid search ---
def search_hybrid(query, k=5):
    p = parse_query(query)
    candidates = set(range(len(docs)))
    notes = []

    entity_hits = set()
    for n in p["players"]:
        entity_hits |= player_idx[n]
    for n in p["teams"]:
        entity_hits |= team_idx[n]
    if entity_hits:
        candidates &= entity_hits

    season_applied = False
    if p["season"]:
        hits = {i for i, d in enumerate(docs) if d["season"] == p["season"]}
        narrowed = candidates & hits
        if narrowed:
            candidates = narrowed
            season_applied = True
        else:
            notes.append(
                f"No documents match {p['season']} for this entity. "
                f"Player data covers {', '.join(PLAYER_SEASONS)} only."
            )

    if not candidates:
        return {"results": [], "parsed": p, "season_applied": False, "notes": notes}

    q = model.encode([query])[0].astype("float32")
    q = q / np.linalg.norm(q)
    cand = sorted(candidates)
    scores = embeddings[cand] @ q
    order = np.argsort(-scores)[:k]

    return {
        "results": [{**docs[cand[i]], "score": float(scores[i])} for i in order],
        "parsed": p,
        "season_applied": season_applied,
        "notes": notes,
    }


# --- Quick check ---
out = search_hybrid("How many points did Nikola Jokić average in 2023-24?", 3)
print(out["results"][0]["player"], out["results"][0]["season"])

Nikola Jokić 2023-24


In [5]:
CORPUS_SCOPE = (
    "NBA regular season data only. "
    f"Player statistics cover {', '.join(PLAYER_SEASONS)}. "
    f"Team statistics cover {', '.join(SEASONS)}. "
    "No playoff data, no contracts, no biographical information, no draft history, "
    "no seasons outside this range."
)

PROMPT = """You are a basketball operations analyst. Answer the question using ONLY the \
statistics in the provided context. 

Dataset scope: {scope}

Rules:
- Use only numbers that appear in the context. Never estimate, infer, or recall from memory.
- If the context does not contain the answer, say exactly what is missing. Do not guess.
- When a document says "across N games in this dataset," mention that the figure is based \
on N games rather than a full season.
- Be concise. Two or three sentences unless the question needs more.
{notes}
Context:
{context}

Question: {question}

Answer:"""


def build_prompt(question, retrieval):
    if not retrieval["results"]:
        context = "(no relevant documents found)"
    else:
        context = "\n\n".join(
            f"[Document {i+1}]\n{r['text']}" for i, r in enumerate(retrieval["results"])
        )

    notes = ""
    if retrieval["notes"]:
        notes = "\nImportant: " + " ".join(retrieval["notes"]) + "\n"

    return PROMPT.format(
        scope=CORPUS_SCOPE,
        notes=notes,
        context=context,
        question=question,
    )

print(build_prompt("How many points did Nikola Jokić average in 2023-24?",
                   search_hybrid("How many points did Nikola Jokić average in 2023-24?", 2)))

You are a basketball operations analyst. Answer the question using ONLY the statistics in the provided context. 

Dataset scope: NBA regular season data only. Player statistics cover 2023-24. Team statistics cover 2023-24, 2024-25. No playoff data, no contracts, no biographical information, no draft history, no seasons outside this range.

Rules:
- Use only numbers that appear in the context. Never estimate, infer, or recall from memory.
- If the context does not contain the answer, say exactly what is missing. Do not guess.
- When a document says "across N games in this dataset," mention that the figure is based on N games rather than a full season.
- Be concise. Two or three sentences unless the question needs more.

Context:
[Document 1]
Nikola Jokić, DEN, 2023-24 regular season. Across 77 games in this dataset, averaged 26.5 points, 12.3 rebounds, 9.0 assists, 1.4 steals, and 0.9 blocks in 34.1 minutes per game. Shot 59.4% from the field, 36.0% from three (2.9 attempts per game), a

In [6]:
def answer(question, k=3, show_context=False):
    retrieval = search_hybrid(question, k)
    prompt = build_prompt(question, retrieval)
    response = generate(prompt)

    return {
        "question": question,
        "answer": response.strip(),
        "sources": [
            {"id": r["id"], "name": r.get("player") or r.get("team"),
             "season": r["season"], "score": round(r["score"], 3)}
            for r in retrieval["results"]
        ],
        "notes": retrieval["notes"],
        "prompt": prompt if show_context else None,
    }

In [7]:
tests = [
    # Tier 1 — simple lookup
    "How many points per game did Nikola Jokić average in 2023-24?",
    # Tier 2 — filtered
    "How did LeBron James perform in road games in 2023-24?",
    # Tier 3 — comparison
    "Who shot better from three in 2023-24, Stephen Curry or Anthony Edwards?",
    # Tier 4 — multi-step (expected to fail)
    "Which players averaged over 25 points while shooting above 50%?",
    # Tier 5 — out of scope
    "What is Nikola Jokić's contract worth?",
    "How many points did Luka Dončić average in 2024-25?",
]

for q in tests:
    r = answer(q)
    print(f"\n{'='*72}\nQ: {q}\n")
    print(f"A: {r['answer']}\n")
    print(f"   sources: {[f'{s['name']} {s['season']}' for s in r['sources']]}")
    if r["notes"]:
        print(f"   notes: {r['notes']}")


Q: How many points per game did Nikola Jokić average in 2023-24?

A: Nikola Jokić averaged 26.5 points per game across 77 games in the 2023-24 regular season.

   sources: ['Nikola Jokić 2023-24']

Q: How did LeBron James perform in road games in 2023-24?

A: In road games, LeBron James averaged 26.7 points across 32 games in this dataset.

   sources: ['LeBron James 2023-24']

Q: Who shot better from three in 2023-24, Stephen Curry or Anthony Edwards?

A: Stephen Curry shot 40.7% from three, while Anthony Edwards shot 35.7% from three.

   sources: ['Stephen Curry 2023-24', 'Anthony Edwards 2023-24', 'Seth Curry 2023-24']

Q: Which players averaged over 25 points while shooting above 50%?

A: Based on the provided statistics, Jalen Williams averaged 36 points in a single game (Document 2), but not across the entire season. However, his season-high score is the highest among the three players, but we cannot confirm his season average is above 25 points.

The only other player with a s

In [8]:
r = answer("What is Nikola Jokić's contract worth?", show_context=True)
print(r["prompt"])
print("\n" + "="*72)
print(r["answer"])

You are a basketball operations analyst. Answer the question using ONLY the statistics in the provided context. 

Dataset scope: NBA regular season data only. Player statistics cover 2023-24. Team statistics cover 2023-24, 2024-25. No playoff data, no contracts, no biographical information, no draft history, no seasons outside this range.

Rules:
- Use only numbers that appear in the context. Never estimate, infer, or recall from memory.
- If the context does not contain the answer, say exactly what is missing. Do not guess.
- When a document says "across N games in this dataset," mention that the figure is based on N games rather than a full season.
- Be concise. Two or three sentences unless the question needs more.

Context:
[Document 1]
Nikola Jokić, DEN, 2023-24 regular season. Across 77 games in this dataset, averaged 26.5 points, 12.3 rebounds, 9.0 assists, 1.4 steals, and 0.9 blocks in 34.1 minutes per game. Shot 59.4% from the field, 36.0% from three (2.9 attempts per game), a

In [9]:
for q in ["Which players averaged over 25 points while shooting above 50%?",
          "What is Nikola Jokić's contract worth?"]:
    r = answer(q)
    print(f"Q: {q}")
    print(f"A: {r['answer']}")
    print(f"   sources: {[s['name'] + ' ' + s['season'] for s in r['sources']]}")
    print("-" * 60)

Q: Which players averaged over 25 points while shooting above 50%?
A: Based on the provided statistics, neither Amen Thompson nor Jalen Williams nor Jalen Green averaged over 25 points.
   sources: ['Amen Thompson 2023-24', 'Jalen Williams 2023-24', 'Jalen Green 2023-24']
------------------------------------------------------------
Q: What is Nikola Jokić's contract worth?
A: Missing. No information on Nikola Jokić's contract is provided in the given context.
   sources: ['Nikola Jokić 2023-24']
------------------------------------------------------------
